# Pre-EDA Health Check & Data Processing

## Import Modules

In [ ]:
import pandas as pd
import numpy as np
from typing import List, Tuple
from collections.abc import Callable

## Read Subsample Data Parquet File

In [ ]:
MODE = 'COLABS'
MOUNT_PATH = ''
INPUT_PATH = ''

if MODE == 'COLABS':
    from google.colab import drive
    drive.mount('/content/drive')
    MOUNT_PATH = '/content/drive/MyDrive'
    INPUT_PATH = '/DSC 288R/Project'
else:
    INPUT_PATH = '/Users/steveg/Downloads'

# Define the path to your Parquet folder, which is 'subsampled_parquet' inside your project folder
parquet_folder_path = f'{MOUNT_PATH}{INPUT_PATH}/subsampled_parquet'
df = pd.read_parquet(parquet_folder_path)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Understand the Attribute Semantics

| Feature Group | Column Name | Description | Data Type | Variable Type |
|---|---|---|---|---|
| User / Game Features | `author_steamid` | The user's SteamID | Integer | NQ |
| User / Game Features | `appid` | The unique app ID for every app on Steam | Integer | NQ |
| User / Game Features | `author_num_games_owned` | Number of games owned by the user | Integer | DQ |
| User / Game Features | `author_num_reviews` | Number of reviews written by the user | Integer | DQ |
| User / Game Features | `author_playtime_forever` | Lifetime playtime tracked in this app, in minutes | Integer | DQ |
| User / Game Features | `author_playtime_last_two_weeks` | Playtime tracked in the past two weeks for this app | Integer | DQ |
| User / Game Features | `author_playtime_at_review` | Playtime when the review was written | Integer | DQ |
| User / Game Features | `author_last_played` | Time when the user last played, represented as Unix time | Integer / Unix timestamp | DQ |
| Review Features | `review` | Text of the written review | String | Text |
| Review Features | `voted_up` | Whether the review was a positive recommendation | Boolean | NQ |
| Review Features | `votes_up` | Number of users who found this review helpful | Integer | DQ |
| Review Features | `votes_funny` | Number of users who found this review funny | Integer | DQ |
| Review Features | `weighted_vote_score` | Helpfulness score | Float | CQ |
| Review Features | `comment_count` | Number of comments posted on this review | Integer | DQ |
| Review Features | `written_during_early_access` | Whether the user posted this review while the game was in Early Access | Boolean | NQ |
| Review Features | `language` | Language the user indicated when authoring the review | String | NQ |
| Data Origin Feature | `timestamp_created` | Date the review was created, represented as Unix timestamp | Integer / Unix timestamp | DQ |
| Data Origin Feature | `timestamp_updated` | Date the review was last updated, represented as Unix timestamp | Integer / Unix timestamp | DQ |

**Variable Type Notes**

| Abbreviation | Meaning |
|---|---|
| NQ | Nominal qualitative |
| DQ | Discrete quantitative |
| CQ | Continuous quantitative |
| Text | Unstructured text feature |


---
---
---

**The following features are discarded, or discarded after data cleaning, because they are not selected for the final churn analysis feature set.**

| Feature Group                  | Column Name             | Description                                                           | Data Type              | Variable Type |
| ------------------------------ | ----------------------- | --------------------------------------------------------------------- | ---------------------- | ------------- |
| Irrelevant / Platform Features | `hidden_in_steam_china` | Whether the review or app is hidden in Steam China                    | Integer / Boolean-like | NQ            |
| Irrelevant / Platform Features | `steam_china_location`  | Steam China location information, represented with Chinese characters | String                 | NQ            |
| Identifier Features            | `recommendationid`      | Unique ID of the recommendation / review record                       | Integer                | NQ            |
| Review Metadata Features       | `language`              | Language the user indicated when authoring the review                 | String                 | NQ            |
| Purchase Metadata Features     | `steam_purchase`        | Whether the user purchased the game on Steam                          | Integer / Boolean-like | NQ            |
| Purchase Metadata Features     | `received_for_free`     | Whether the user indicated that they received the app for free        | Integer / Boolean-like | NQ            |

These variables are excluded for the following reasons:

* `hidden_in_steam_china` is not useful for churn analysis because it is a region/platform visibility flag rather than a behavioral user feature.
* `steam_china_location` is not useful for churn analysis because most values are missing, and the available values appear too limited to provide meaningful predictive information.
* `recommendationid` is a record identifier. It is useful for database management, but it should not be treated as a predictive feature.
* `language` is removed after data cleaning because the analysis focuses only on English reviews. However, it may still be useful during cleaning to check whether reviews labeled as English contain non-Latin text.
* `steam_purchase` and `received_for_free` are excluded for now because their combined meaning is ambiguous. For example, `steam_purchase = 1` and `received_for_free = 1` may seem contradictory unless Steam treats free installations or free apps as Steam purchases. Because of this uncertainty, these variables are set aside unless more documentation or API evidence is found.


## Health Check - Dataset Structure

In [ ]:
## Shape
# (# of rows, # of columns)
display(df.shape)
print('\n---------------------------------------\n')

# Total cells
display(df.size)
print('\n---------------------------------------\n')

## Concise summary on columns details & memory usage
df.info()

(2105211, 18)


---------------------------------------



37893798


---------------------------------------

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2105211 entries, 0 to 2105210
Data columns (total 18 columns):
 #   Column                          Dtype  
---  ------                          -----  
 0   author_steamid                  int64  
 1   appid                           float64
 2   author_num_games_owned          int32  
 3   author_num_reviews              int32  
 4   author_playtime_forever         int32  
 5   author_playtime_last_two_weeks  int32  
 6   author_playtime_at_review       int32  
 7   author_last_played              int64  
 8   review                          object 
 9   voted_up                        object 
 10  votes_up                        float64
 11  votes_funny                     float64
 12  weighted_vote_score             float32
 13  comment_count                   float64
 14  written_during_early_access     object 
 15  language                        object 
 16  timestamp_created             

Observations:
- The dataframe consists of ~1.4 M rows and 17 columns. It also occupied ~150 MB of memory.
- Pandas inferred some column types differently from our intended schema.
Several integer-like columns were inferred as non-nullable numeric types.
    - Columns intended to be integer values may appear as `int32` or `float64`, even though the dataset contains null values. Since standard NumPy integer types such as `int32` are not nullable, these columns should use Pandas nullable integer types such as `Int32` or `Int64`.
    - `appid` was inferred as `float64`, but it should represent an app identifier.Since appid is a unique Steam app ID, it should be stored as an integer-like identifier, preferably `Int32` if missing values are possible.
    - `voted_up` and `written_during_early_access` were inferred as numeric values instead of `boolean`.These columns represent true/false conditions, so they should be converted to Pandas nullable Boolean type, boolean
    - `timestamp_updated` was inferred as a floating-point column, but it should be an integer Unix timestamp.

Suggestions for Data Processing/Feature Engineering:
- *(Feature Engineering)* Convert all `int32` to `Int32` OR `int64` to `Int64` to improve the robustness to null values
- *(Feature Engineering)* Convert all identifier & temporal columns to be `Int64`
- *(Feature Engineering)* Convert all categorical columns to be `Boolean`
- *(Feature Engineering)* Ensure `weighted_score` column to be `Float64`

## Feature Engineering - Type Conversion

In [ ]:
def organize_columns() -> List[Tuple[str, str]]:
    # Define type to convert
    int32_cols = [
        "author_num_games_owned",
        "author_num_reviews",
        "author_playtime_forever",
        "author_playtime_last_two_weeks",
        "author_playtime_at_review",
        "votes_up",
        "comment_count",
    ]

    int64_cols = [
        "author_steamid",
        "appid",
        "author_last_played",
        "timestamp_created",
        "timestamp_updated",
        "votes_funny",
    ]

    float32_cols = [
        "weighted_vote_score"
    ]

    bool_cols = [
        "voted_up",
        "written_during_early_access"
    ]

    datetime_cols = [
        "timestamp_created",
        "timestamp_updated",
        "author_last_played"
    ]

    text_cols = [
        "review"
    ]

    # Organize it to config map
    type_to_convert = []
    type_to_convert.extend([(col, 'Int32') for col in int32_cols])
    type_to_convert.extend([(col, 'Int64') for col in int64_cols])
    type_to_convert.extend([(col, 'Float32') for col in float32_cols])
    type_to_convert.extend([(col, 'boolean') for col in bool_cols])
    type_to_convert.extend([(col, 'datetime') for col in datetime_cols])
    type_to_convert.extend([(col, 'string') for col in text_cols])

    return type_to_convert

def type_conversion(
    df: pd.DataFrame,
    map_func: Callable[[], List[Tuple[str, str]]]
) -> pd.DataFrame:
    df = df.copy()

    # Convert types using astype(), to_datetime()
    for (col_name, col_type) in map_func():
        if col_type.lower() == 'datetime':
            df[col_name] = pd.to_datetime(df[col_name], unit="s", errors="coerce")
        else:
            df[col_name] = df[col_name].astype(col_type)

    return df

In [ ]:
df = type_conversion(df, organize_columns)
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2105211 entries, 0 to 2105210
Data columns (total 18 columns):
 #   Column                          Dtype        
---  ------                          -----        
 0   author_steamid                  Int64        
 1   appid                           Int64        
 2   author_num_games_owned          Int32        
 3   author_num_reviews              Int32        
 4   author_playtime_forever         Int32        
 5   author_playtime_last_two_weeks  Int32        
 6   author_playtime_at_review       Int32        
 7   author_last_played              datetime64[s]
 8   review                          string       
 9   voted_up                        boolean      
 10  votes_up                        Int32        
 11  votes_funny                     Int64        
 12  weighted_vote_score             Float32      
 13  comment_count                   Int32        
 14  written_during_early_access     boolean      
 15  language       

## Health Check - Completeness

In [ ]:
## Compeleteness helps us investigate missing value and if we have useful
## features for prediction problems

# 1. Identify missing values per column
print("1. Identify missing values per column")
missing_values = df.isnull().sum()
print(
    df.isnull()
        .sum()[missing_values > 0]
        .sort_values(ascending=False)
)
print('\n---------------------------------------\n')

# 2. Calculate percentage of missing values
print("2. Calculate percentage of missing values")
percentage_missing_values = (df.isnull().sum() * 100/ len(df))
print(
    percentage_missing_values[percentage_missing_values > 0]
        .sort_values(ascending=False)
)

# 3. Check if the data is enough for ML tasks and are the features
# for the analysis

# Based on the previous observation, we know that we have 1.4M data points,
# which is enough for classical machine learning task. Since this is only 2%
# of the full dataset, we ensure that we are doing big data analytics.
# Regarding churn analysis, since we have user and time behavior on review,
# it should give us a quite good idea on player behavior on using Steam.

1. Identify missing values per column
voted_up                       50
timestamp_updated              33
written_during_early_access    27
votes_up                       27
weighted_vote_score            26
votes_funny                    25
comment_count                  23
appid                           3
dtype: int64

---------------------------------------

2. Calculate percentage of missing values
voted_up                       0.002375
timestamp_updated              0.001568
written_during_early_access    0.001283
votes_up                       0.001283
weighted_vote_score            0.001235
votes_funny                    0.001188
comment_count                  0.001093
appid                          0.000143
dtype: float64


Observations:
- All columns with missing values are only between 0 ~ 0.002% of all data, which could be negligible for the ML task. However, we are still interested in how the columns with missing value possibly depends on other columns in later EDA report

Suggestions for Data Processing/Feature Engineering:
- *(Data Processing)* Despite missing proportion is so small comparing the full dataset, we don't know which 0.2% are they missing. I will expect a possible distribution change due to uncertainty; practically speaking however, If the missing rows are not concentrated in important user, game, time, or target-related groups, removal is unlikely to meaningfully affect the overall distribution. We decided to perform list deletion for now.

## Processing - Missing Value Removal

In [ ]:
df = df.dropna()
print(
    df.isnull()
        .sum()[missing_values > 0]
        .sort_values(ascending=False)
)

appid                          0
voted_up                       0
votes_up                       0
votes_funny                    0
weighted_vote_score            0
comment_count                  0
written_during_early_access    0
timestamp_updated              0
dtype: int64


## Health Check - Consistency & Validity

In [ ]:
## Consistency helps us question "Do columns agree with each other logically"?

# 1. timestamp_created should usually be <= timestamp_updated
print("1. timestamp_created should usually be <= timestamp_updated")
print(df["timestamp_created"].astype('int64').sum() <= df["timestamp_updated"].astype('int64').sum())
print('\n---------------------------------------\n')

# 2. playtime_at_review & playtime_last two weeks should be less than lifetime playtime
print("2. playtime_at_review & playtime_last two weeks should be less than lifetime playtim")
print(df["author_playtime_at_review"].sum() <= df["author_playtime_forever"].sum())
print(df["author_playtime_last_two_weeks"].sum() <= df["author_playtime_forever"].sum())
print('\n---------------------------------------\n')

# 3. all user in the dataset should have at least 1 numbers of review
print("3. all user in the dataset should have at least 1 numbers of review")
invalid_user_lst = np.nonzero(df.groupby('author_steamid')['author_num_reviews'].min() == 0)[0]
print(f"List of user should be concerns:  {invalid_user_lst}")
print('\n---------------------------------------\n')

# 4. author_playtime_forever at later review time should not be lower than earlier review time
print("4. author_playtime_forever at later review time should not be lower than earlier review time")
# Sort and shift to previous playtime_forever if possible
df_sorted = (
    df.sort_values(["author_steamid", "appid", "timestamp_created"])
        .copy()
        .assign(**{"prev_playtime_forever": lambda df: (
            df.groupby(["author_steamid", "appid"])
            ["author_playtime_forever"]
            .shift(1)
        )})
)
# Compare previous to current playtime forever
decreasing_playtime = df_sorted[
    df_sorted["author_playtime_forever"] < df_sorted["prev_playtime_forever"]
]
print(f"Numbers of instance violating the rule: {len(decreasing_playtime)}")
print('\n---------------------------------------\n')

## Validity helps us question "Do columns within reasonable range"?

# 5. Count-like columns should not be negative
print("5. Count-like columns should not be negative")
count_cols = [
    "author_num_games_owned",
    "author_num_reviews",
    "author_playtime_forever",
    "author_playtime_last_two_weeks",
    "author_playtime_at_review",
    "votes_up",
    "votes_funny",
    "comment_count",
]
print((df[count_cols] < 0).sum())
print('\n---------------------------------------\n')

# 6. weighted_vote_score should be between 0 and 1
print("6. weighted_vote_score should be between 0 and 1")
print(np.all(0 <= df['weighted_vote_score']) <= 1)

1. timestamp_created should usually be <= timestamp_updated
True

---------------------------------------

2. playtime_at_review & playtime_last two weeks should be less than lifetime playtim
True
True

---------------------------------------

3. all user in the dataset should have at least 1 numbers of review
List of user should be concerns:  []

---------------------------------------

4. author_playtime_forever at later review time should not be lower than earlier review time
Numbers of instance violating the rule: 0

---------------------------------------

5. Count-like columns should not be negative
author_num_games_owned            0
author_num_reviews                0
author_playtime_forever           0
author_playtime_last_two_weeks    0
author_playtime_at_review         0
votes_up                          0
votes_funny                       0
comment_count                     0
dtype: Int64

---------------------------------------

6. weighted_vote_score should be between 0 a

Observations: All above scenarios are tested to have consistency

## Health Check - Anomaly & Outliers

In [ ]:
## Anomaly indicates suspicious value, not automatically wrong

# 1. Check for outlier value using summary statistics
print("1. Check for outlier value using summary statistics\n")
numeric_cols = [
    "author_num_games_owned",
    "author_num_reviews",
    "author_playtime_forever",
    "author_playtime_last_two_weeks",
    "author_playtime_at_review",
    "votes_up",
    "votes_funny",
    "weighted_vote_score",
    "comment_count",
]
display(df[numeric_cols].describe(percentiles=[.01, .05, .25, .5, .75, .95, .99]).T)
print('\n---------------------------------------\n')

# 2. Create report regarding anomaly report for suspicious features (1) votes_funny artifact
print("2. Create report regarding anomaly report for suspicious features (1) votes_funny artifact; (2) author_playtime_last_two_weeks \n")

# Define the extreme limits
df_anomaly = df.copy()
VOTES_FUNNY_ARTIFACT_THRESHOLD = 4_000_000_000
VOTES_FUNNY_MAX = 2**32 - 1  # This equals to the max value of votes_funny from summary statistics, but in compact form
MAX_2W_MINUTES = 14 * 24 * 60

# Create new features representing flag/no flag suspicious behavior
df_anomaly = (
    df_anomaly
        .assign(**{
            'votes_funny_is_uint32_artifact': lambda df: df['votes_funny'] >= VOTES_FUNNY_ARTIFACT_THRESHOLD,
            'votes_funny_near_exact_uint32_artifact_max': lambda df: df['votes_funny'].between(VOTES_FUNNY_ARTIFACT_THRESHOLD, VOTES_FUNNY_MAX, inclusive='left'),
            'playtime_2w_near_limit': lambda df: df['author_playtime_last_two_weeks'] >= 0.90 * MAX_2W_MINUTES,
            'playtime_2w_exceeds_limit': lambda df: df['author_playtime_last_two_weeks'] > MAX_2W_MINUTES
        })
        .drop(columns=df.columns)
)

# Create anomaly report
display(
    pd
        .DataFrame({
            "flag_count": df_anomaly[df_anomaly.columns].sum(),
            "flag_rate": df_anomaly[df_anomaly.columns].mean()
        })
        .sort_values("flag_count", ascending=False)
)

print('\n---------------------------------------\n')

# 3. Check for instances where votes_funny reached max value
print("3. Check for instances where votes_funny reached max value\n")
display(
    df[df["votes_funny"] == VOTES_FUNNY_MAX]
        .sort_values("votes_funny", ascending=False)
        [numeric_cols]
)
print('\n---------------------------------------\n')

# 4. Check for instances where votes_funny near max value
print("4. Check for instance where votes_funny near max value")
display(
    df[df["votes_funny"].between(VOTES_FUNNY_ARTIFACT_THRESHOLD, VOTES_FUNNY_MAX, inclusive='left')]
        .sort_values("votes_funny", ascending=False)
        [numeric_cols]
)
print('\n---------------------------------------\n')

# 5. Find the largest vote_funny value that follows right after ARTIFACT_THRESHOLD, compute its propoertion comparing to max value
# If it's between 60% - 95%, it will indicate broader range of artifact from API and we will need to dive deeper
print("5. Find the largest vote_funny value that follows right after ARTIFACT_THRESHOLD, compute its propoertion comparing to max value")
proportion = (
    (df[df["votes_funny"] < VOTES_FUNNY_ARTIFACT_THRESHOLD]
        .sort_values("votes_funny", ascending=False)
        .reset_index(drop=True)
        .loc[0, "votes_funny"]
    ) / VOTES_FUNNY_MAX * 100
)
print(f"This value is {proportion}% of VOTES_FUNNY_MAX value!")

del df_anomaly, proportion

1. Check for outlier value using summary statistics



,count,mean,std,min,1%,5%,25%,50%,75%,95%,99%,max
author_num_games_owned,2105161.0,114.226471,371.015141,0.0,0.0,0.0,0.0,2.0,98.0,503.0,1442.0,20481.0
author_num_reviews,2105161.0,19.62353,55.164638,1.0,1.0,1.0,2.0,6.0,16.0,75.0,237.0,941.0
author_playtime_forever,2105161.0,15850.705158,46324.619402,0.0,0.0,40.0,586.0,2373.0,10472.0,79067.0,202435.8,3815539.0
author_playtime_last_two_weeks,2105161.0,63.881836,473.854041,0.0,0.0,0.0,0.0,0.0,0.0,175.0,1669.0,20156.0
author_playtime_at_review,2105161.0,7163.961216,24329.299949,0.0,0.0,24.0,272.0,928.0,3746.0,35408.0,105216.8,2835307.0
votes_up,2105161.0,2.117855,33.477302,0.0,0.0,0.0,0.0,0.0,1.0,5.0,30.0,14583.0
votes_funny,2105161.0,73448.221202,17760894.856098,0.0,0.0,0.0,0.0,0.0,0.0,1.0,8.0,4294967295.0
weighted_vote_score,2105161.0,0.179188,0.250308,0.0,0.0,0.0,0.0,0.0,0.492399,0.549365,0.70266,0.994626
comment_count,2105161.0,0.103504,1.624198,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.0,1131.0



---------------------------------------

2. Create report regarding anomaly report for suspicious features (1) votes_funny artifact; (2) author_playtime_last_two_weeks 



,flag_count,flag_rate
playtime_2w_near_limit,221,0.000105
votes_funny_is_uint32_artifact,36,0.000017
votes_funny_near_exact_uint32_artifact_max,8,0.000004
playtime_2w_exceeds_limit,0,0.0



---------------------------------------

3. Check for instances where votes_funny reached max value



,author_num_games_owned,author_num_reviews,author_playtime_forever,author_playtime_last_two_weeks,author_playtime_at_review,votes_up,votes_funny,weighted_vote_score,comment_count
4177,1152,438,1758,0,1387,1,4294967295,0.409136,0
54995,918,62,2842,0,1035,0,4294967295,0.49647,0
69086,0,11,2175,0,1077,4,4294967295,0.522334,2
136680,33,1,26,0,12,3,4294967295,0.502463,0
137854,0,41,6847,0,1215,76,4294967295,0.728008,0
175586,47,4,245899,0,3389,3,4294967295,0.548872,0
237240,96,3,141,0,119,13,4294967295,0.168718,5
240336,22,3,335,0,300,0,4294967295,0.476868,0
592440,827,60,2191,0,1192,19,4294967295,0.621283,1
612048,149,5,14697,0,14516,8,4294967295,0.605863,0



---------------------------------------

4. Check for instance where votes_funny near max value


,author_num_games_owned,author_num_reviews,author_playtime_forever,author_playtime_last_two_weeks,author_playtime_at_review,votes_up,votes_funny,weighted_vote_score,comment_count
207333,0,1,160,0,0,1,4294967294,0.502488,0
399424,244,16,3474,0,2741,2,4294967294,0.527406,0
965323,89,25,35110,0,18448,1,4294967294,0.502488,0
1295206,73,2,81434,0,11682,6,4294967294,0.474069,3
1732816,0,7,53478,0,14113,6,4294967294,0.609823,0
1087214,318,13,82215,0,32603,0,4294967293,0.464773,0
257117,111,4,1029,0,956,4,4294967291,0.404808,0
2104470,0,1,25882,0,7381,1,4294967290,0.502488,1



---------------------------------------

5. Find the largest vote_funny value that follows right after ARTIFACT_THRESHOLD, compute its propoertion comparing to max value
This value is 0.00055241398526179% of VOTES_FUNNY_MAX value!


Observations:
- `author_num_games_owned`: This is very skewed, but not automatically wrong. Some Steam users can own many games.
- `author_num_reviews`: This looks reasonable. Heavy reviewers exist.
- `author_playtime_forever`: Extreme outlier, but not automatically invalid.
- ***`author_playtime_last_two_weeks`: max = 20156 minutes ≈ 335.9 hours, which is almost exactly 2 weeks, it's physically possible if the player did not exit the steam/game (depends on how they measure playtime) and its activities is recorded continuously***
- `author_playtime_at_review`: Outliers are expected, but playtime_at_review > playtime_forever would be a consistency issue.
- `votes_up`: Not invalid, but highly sparse and heavy-tailed.
- ***`votes_funny`: Major anomaly --> likely API data artifact or an unsigned integer overflow***
- `weighted_vote_score`: No obvious range issue.
- `comment_count`: Very sparse, strongly right-skewed. Most reviews have no comments, but a few may have large discussions.


Suggestions for Data Processing/Feature Engineering:
- *(Feature Engineering)* Regarding `author_playtime_last_two_weeks`: Only 31 records are near the physical maximum of two-week playtime, representing roughly 0.004% of the sample. These values are suspicious but possible because Steam playtime may reflect application runtime rather than active human gameplay. I therefore flag these records for quality tracking rather than remove them.

- *(Data Processing)* Regarding `votes_funny`:  Several values are equal or very close to 2^32 - 1, the maximum unsigned 32-bit integer. These values are separated from the plausible range by a large discontinuity: normal high values are only in the thousands, while the suspicious values jump to over 4 billion. Because these records also have low votes_up and only moderate weighted_vote_score, I treat them as likely integer/API artifacts rather than genuine engagement counts. Thus I will remove the suspicious values.

- *(Feature Engineering)* For skewed count variables, apply `log1p` transformation:
  - `log_votes_up = log1p(votes_up)`
  - `log_votes_funny = log1p(votes_funny)`
  - `log_comment_count = log1p(comment_count)`
  - `log_playtime_last_two_weeks = log1p(author_playtime_last_two_weeks)`


## Processing - Remove Invalid Outlier

In [ ]:
# Remove rows with vote funny with artifact values
df = df[df["votes_funny"] < VOTES_FUNNY_ARTIFACT_THRESHOLD]

## Health Check - Noise

In [ ]:
## Noises are values that are valid but possibly not useful & unstable

# 1. Check for columns with mostly zeros
print("# 1. Check for columns with mostly zeros")
categorical_cols = [
    "voted_up",
    "written_during_early_access"
]
display(
    (df[numeric_cols + categorical_cols] == 0)
        .mean()
        .sort_values()
)

# 1. Check for columns with mostly zeros


,0
author_num_reviews,0.0
author_playtime_forever,0.01495
author_playtime_at_review,0.016125
voted_up,0.136853
author_num_games_owned,0.495344
weighted_vote_score,0.655016
votes_up,0.694739
votes_funny,0.877833
written_during_early_access,0.891307
author_playtime_last_two_weeks,0.90153


Observations:
- Several columns are **highly zero-inflated**, meaning most records contain `0` rather than a positive value.
- `comment_count` is the sparsest numeric feature:
  - About **95.8%** of rows have `comment_count = 0`.
  - This suggests most reviews receive no comments.
- `author_playtime_last_two_weeks` is also highly sparse:
  - About **90.8%** of rows have `0` recent playtime.
  - This may be important for churn because recent inactivity is directly related to churn behavior.
- `votes_funny` is highly sparse:
  - About **87.7%** of rows have `0`.
  - This feature may still contain signal, but mostly as a rare engagement indicator.
- `votes_up` is moderately to highly sparse:
  - About **69.2%** of rows have `0`.
  - The raw count may be less useful than a binary indicator or transformed value.
- `weighted_vote_score` has many zeros:
  - About **65.2%** of rows have `0`.
  - This suggests the score may be unavailable, uninformative, or only meaningful when reviews receive enough voting activity.
- `author_num_games_owned` has about **50.0%** zeros:
  - This is suspicious from a behavioral perspective because many Steam users likely own at least one game.
  - The underlying data-generating mechanism for `author_num_games_owned = 0` cannot be fully determined from the dataset alone. Although the value is observed as zero, it may represent true zero ownership, unavailable/private ownership metadata, API default behavior, free-to-play access patterns, or a mismatch between review time and data collection time. Therefore, this field is treated as “zero recorded owned games” rather than literal proof that the user owns no games.

Suggestions for Data Processing/Feature Engineering:
  - *(Feature Engineering)*: For highly zero-inflated count columns, consider creating binary indicator features:
    - `has_comments = comment_count > 0`
    - `has_votes_up = votes_up > 0`
    - `has_funny_votes = votes_funny > 0`
    - `has_recent_playtime = author_playtime_last_two_weeks > 0`
    - `has_weighted_vote_score = weighted_vote_score > 0`



## Health Check - Uniqueness & Duplicates

In [ ]:
## Uniqueness of catgorical features tells us insight of categories, while for numerical features,
## it indicates potential few_unique_values_per_column issue or possibility for discretization to a categorical type

# 1. Numbers of unique values for categorical/numerical & ID columns
print("1. Numbers of unique values for categorical/numerical & ID columns")
print(
    df[categorical_cols]
        .nunique()
        .sort_values()
)
print()
print(
    df[numeric_cols]
        .nunique()
        .sort_values()
)
print()
print(
    df[["author_steamid", "appid"]]
        .nunique()
        .sort_values()
)
print('\n---------------------------------------\n')

## Duplicate tells us how many rows has exact duplicates and how frequent would a user to review the same game

# 2. Numbers of duplicate found in dataframe
print("2. Numbers of duplicate found in dataframe")
exact_dup_mask = df.duplicated(keep=False)
print(exact_dup_mask.sum())
print('\n---------------------------------------\n')

# 3. Does the same user appear to have created multiple review records for the same game at the same time?
print("3. Does the same user appear to have created multiple review records for the same game at the same time?")
key_cols = ["author_steamid", "appid", "timestamp_created"]
combo_dup_mask = df.duplicated(subset=key_cols, keep=False)
print(combo_dup_mask.sum())

1. Numbers of unique values for categorical/numerical & ID columns
voted_up                       2
written_during_early_access    2
dtype: int64

comment_count                        130
author_num_reviews                   302
votes_funny                          681
votes_up                             985
author_num_games_owned              2549
author_playtime_last_two_weeks      7182
author_playtime_at_review          98171
author_playtime_forever           152108
weighted_vote_score               231940
dtype: int64

appid              40418
author_steamid    722995
dtype: int64

---------------------------------------

2. Numbers of duplicate found in dataframe
30

---------------------------------------

3. Does the same user appear to have created multiple review records for the same game at the same time?
44


In [ ]:
# 4. Inspect more details ("author_steamid", "appid", "timestamp_created") regarding instances in #3
print("4. Inspect more details from other columns regarding instance from last part")
combo_dup = (
    # Count distinct full rows inside each duplicate-key group
    df[combo_dup_mask].groupby(key_cols, dropna=False)
    .apply(lambda g: g.drop_duplicates().shape[0])  # First de-duplicate and then do row count
    .rename("n_distinct_full_rows")
    .reset_index()
    # Keep key groups where duplicate-key rows are not all exact copies
    .query("n_distinct_full_rows > 1")
    # Reduce compution when joining
    [key_cols]
    # Join to filter combo duplicate that don't contain exact duplicate in the final result
    .merge(
        df[combo_dup_mask],
        on=key_cols,
        how="inner"
    )
    # Format nicely
    .sort_values(key_cols)
)
display(combo_dup)
print("Rows involved:", len(combo_dup))
print('\n---------------------------------------\n')

# 5. Print out the duplicate reviews
print("5. Print out the duplicate reviews\n")
for review in combo_dup['review']: print(review)

4. Inspect more details from other columns regarding instance from last part


/tmp/ipykernel_199266/4104670532.py:6: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: g.drop_duplicates().shape[0])  # First de-duplicate and then do row count


,author_steamid,appid,timestamp_created,author_num_games_owned,author_num_reviews,author_playtime_forever,author_playtime_last_two_weeks,author_playtime_at_review,author_last_played,review,voted_up,votes_up,votes_funny,weighted_vote_score,comment_count,written_during_early_access,language,timestamp_updated
0,76561198024366938,255710,2015-06-21 20:17:18,0,1,854,0,146,2023-05-13 14:04:01,"MacBook Pro (Retina, 15', Mitte 2014), 2,5 GHz...",False,0,0,0.0,0,False,english,2015-06-21 20:17:18
1,76561198024366938,255710,2015-06-21 20:17:18,0,1,854,0,146,2023-05-13 14:04:01,"MacBook Pro (Retina, 15', Mitte 2014), 2,5 GHz...",False,0,0,0.0,0,False,german,2015-06-21 20:17:18
2,76561198024366938,255710,2015-06-21 20:17:18,0,1,854,0,146,2023-05-13 14:04:01,"MacBook Pro (Retina, 15', Mitte 2014), 2,5 GHz...",False,0,0,0.0,0,False,german,2015-06-21 20:17:18
3,76561198024366938,255710,2015-06-21 20:17:18,0,1,854,0,146,2023-05-13 14:04:01,"MacBook Pro (Retina, 15', Mitte 2014), 2,5 GHz...",False,0,0,0.0,0,False,german,2015-06-21 20:17:18
4,76561198030127910,201810,2014-06-05 13:32:15,0,30,7173,0,2176,2017-10-29 11:58:32,"Амосфернейший шутер, давно не было коридороног...",True,1,1,0.52381,0,False,russian,2014-06-14 13:01:04
5,76561198030127910,201810,2014-06-05 13:32:15,0,30,7173,0,664,2017-10-29 11:58:32,"Амосфернейший шутер, давно не было коридороног...",True,0,0,0.0,0,False,russian,2014-06-05 13:32:15
6,76561198074796884,730,2014-03-14 19:52:47,636,1,93443,0,3637,2023-10-08 10:49:15,best game evarrrrrrrrrrr!!!!!!!!!!!!!!!!!!!!!!...,True,3,0,0.565217,0,False,german,2014-03-14 19:52:47
7,76561198074796884,730,2014-03-14 19:52:47,636,1,93443,0,3637,2023-10-08 10:49:15,best game evarrrrrrrrrrr!!!!!!!!!!!!!!!!!!!!!!...,True,1,0,0.52381,0,False,german,2014-03-14 19:52:47
8,76561198074796884,730,2014-03-14 19:52:47,636,1,93443,0,3637,2023-10-08 10:49:15,best game evarrrrrrrrrrr!!!!!!!!!!!!!!!!!!!!!!...,True,0,0,0.0,0,False,german,2014-03-14 19:52:47
9,76561198074796884,730,2014-03-14 19:52:47,636,1,93443,0,3637,2023-10-08 10:49:15,best game evarrrrrrrrrrr!!!!!!!!!!!!!!!!!!!!!!...,True,0,0,0.0,0,False,english,2014-03-14 19:52:47


Rows involved: 24

---------------------------------------

5. Print out the duplicate reviews

MacBook Pro (Retina, 15', Mitte 2014), 2,5 GHz Intel Core i7, 16 GB 1600 MHz DDR3, NVIDIA GeForce GT 750M 2048 MB;  Setting the Graphics Settings all to low and using a Full HD Resolution my Fans are going crazy and the MacBook is heating up. This is a really awful performance. I can play games like Diablo 3 with amazing Graphics and the fans will remain absolutely silent. I wan't my money back!
MacBook Pro (Retina, 15', Mitte 2014), 2,5 GHz Intel Core i7, 16 GB 1600 MHz DDR3, NVIDIA GeForce GT 750M 2048 MB;  Setting the Graphics Settings all to low and using a Full HD Resolution my Fans are going crazy and the MacBook is heating up. This is a really awful performance. I can play games like Diablo 3 with amazing Graphics and the fans will remain absolutely silent. I wan't my money back!
MacBook Pro (Retina, 15', Mitte 2014), 2,5 GHz Intel Core i7, 16 GB 1600 MHz DDR3, NVIDIA GeForce GT 750M 

In [ ]:
# 6. Create report on metadata difference on same (author ID, game ID, review date created)
print("6. Create report on metadata difference on same (author ID, game ID, review date created)")

# Explanation see #4
n_distinct_full_rows = (
    df[combo_dup_mask].groupby(key_cols, dropna=False)
        .apply(lambda g: g.drop_duplicates().shape[0])
        .rename("n_distinct_full_rows")
        .reset_index()
)

diagnostic_report = (
    # Compute various stats inside each duplicate-key group
    df[combo_dup_mask].groupby(key_cols, dropna=False)
        .agg(
            n_rows=("appid", "size"),
            n_distinct_full_rows=("appid", lambda s: None),  # placeholder, see below
            n_review_texts=("review", "nunique"),
            n_languages=("language", "nunique"),
            n_timestamp_updated=("timestamp_updated", "nunique"),
            n_playtime_at_review=("author_playtime_at_review", "nunique"),
            n_votes_up=("votes_up", "nunique"),
            n_votes_funny=("votes_funny", "nunique"),
            n_weighted_vote_score=("weighted_vote_score", "nunique"),
        )
        .reset_index()
        # Join to filter combo duplicate that don't contain exact duplicate in the final result
        .merge(
            n_distinct_full_rows,
            on=key_cols,
            how="left"
        )
        # Drop extra columns
        .drop(columns=["n_distinct_full_rows_x"])
        .rename(columns={"n_distinct_full_rows_y": "n_distinct_full_rows"})
        .sort_values("n_rows", ascending=False)
)
display(diagnostic_report)
del diagnostic_report, n_distinct_full_rows

6. Create report on metadata difference on same (author ID, game ID, review date created)


/tmp/ipykernel_199266/3492186768.py:7: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: g.drop_duplicates().shape[0])


,author_steamid,appid,timestamp_created,n_rows,n_review_texts,n_languages,n_timestamp_updated,n_playtime_at_review,n_votes_up,n_votes_funny,n_weighted_vote_score,n_distinct_full_rows
2,76561198024366938,255710,2015-06-21 20:17:18,4,1,2,1,1,1,1,1,2
7,76561198074796884,730,2014-03-14 19:52:47,4,1,2,1,1,3,1,3,4
15,76561198151549673,730,2015-02-15 09:20:43,4,1,1,1,1,2,1,2,2
14,76561198137371138,243930,2014-06-04 17:01:20,4,1,1,1,1,1,2,2,2
12,76561198133914594,272350,2014-06-04 13:50:35,4,1,1,1,1,1,1,1,1
5,76561198065215070,258160,2015-02-28 12:17:14,3,1,1,1,1,1,1,1,1
1,76561198013036300,104900,2014-03-14 18:15:50,3,1,1,1,1,1,1,1,1
3,76561198030127910,201810,2014-06-05 13:32:15,2,2,1,2,2,2,2,2,2
6,76561198070870953,206190,2014-06-17 23:12:00,2,1,1,1,1,1,1,1,1
4,76561198057462316,730,2014-03-14 17:57:19,2,1,1,1,1,1,1,1,1


Observations:
- The uniqueness checks show expected behavior. Boolean categorical variables each contain two unique values, matching their expected binary structure. Numeric variables show different levels of cardinality: count-based variables such as comment_count, votes_funny, votes_up, and author_num_reviews have fewer unique values, while playtime and weighted score variables have much higher cardinality. This pattern is expected and does not indicate a data quality problem by itself.

- The ID columns have high cardinality, with many unique users and games. This is expected because author_steamid and appid are entity identifiers. They should not be treated as ordinary continuous numeric variables. Instead, they should be used for grouping, aggregation, deduplication, and possible user-level or game-level feature engineering.

- The duplicate check found 21 exact duplicate rows, indicating a small amount of fully redundant data.

- The composite review-key check using author_steamid, appid, and timestamp_created found 29 duplicate-key records. After inspecting duplicate-key records that are not exact duplicates, 12 rows across 6 duplicate-key groups show differences in other fields. Some duplicate-key rows differ in language labels despite having identical review text, suggesting possible source-level language classification noise. Other groups differ in votes_up, votes_funny, weighted_vote_score, author_playtime_at_review, review text, or timestamp_updated. This suggests that some records may represent updated review versions or time-varying review engagement metrics rather than independent review events.

Suggestions for Data Processing/Feature Engineering:

- *(Data Processing)* Remove exact duplicate, because they provide no additional information and may bias count-based summaries.

- *(Data Processing)* Remaining duplicate review-key records should be resolved using a deterministic rule, such as keeping the row with the latest timestamp_updated for each author_steamid-appid-timestamp_created group. This avoids double-counting the same review event while preserving the most recent available version.

## Processing - Remove Duplicates

In [ ]:
# Remove exact duplicates
df = df.drop_duplicates(keep='first')

# Remove older (author ID, game ID, timestamp created) combo duplicate and keep the most recent one
df = df.sort_values("timestamp_updated").drop_duplicates(subset=key_cols, keep="last")

## Save Cleaned File

In [ ]:
OUTPUT_PATH = "COLABS"

if MODE == "COLABS":
    OUTPUT_PATH = "/DSC 288R/Project"

elif MODE == "EXPANSE":
    OUTPUT_PATH = "/expanse/lustre/scratch/bguo3/temp_project/steam_reviews"

else:
    OUTPUT_PATH = "/Users/steveg/Downloads"


# Define output Parquet folder/file path
parquet_output_path = f"{MOUNT_PATH}{OUTPUT_PATH}/cleaned_sample.parquet"

# Export Pandas DataFrame to Parquet
df.to_parquet(
    parquet_output_path,
    index=False,
    engine="pyarrow"
)

print(f"Saved Parquet file to: {parquet_output_path}")

Saved Parquet file to: /content/drive/MyDrive/DSC 288R/Project/cleaned_sample.parquet
